In [3]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
from gensim.models import Word2Vec, FastText
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')


def get_tfidf_features(texts):
    """Returns TF-IDF features (Sparse Matrix)"""
    tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=2500, sublinear_tf=True)
    return tfidf.fit_transform(texts)

def get_word2vec_features(texts, vector_size=100):
    """Returns averaged Word2Vec vectors (Dense)"""
    tokenized = [word_tokenize(t.lower()) for t in texts]
    model = Word2Vec(tokenized, vector_size=vector_size, window=5, min_count=2, workers=4, epochs=10)
    return np.array([np.mean([model.wv[w] for w in tokens if w in model.wv], axis=0) 
                     if any(w in model.wv for w in tokens) else np.zeros(vector_size) 
                     for tokens in tokenized])

def get_fasttext_features(texts, vector_size=100):
    """Returns averaged FastText vectors (Dense)"""
    tokenized = [word_tokenize(t.lower()) for t in texts]
    model = FastText(tokenized, vector_size=vector_size, window=5, min_count=2, workers=4, epochs=10)
    return np.array([np.mean([model.wv[w] for w in tokens if w in model.wv], axis=0) 
                     if any(w in model.wv for w in tokens) else np.zeros(vector_size) 
                     for tokens in tokenized])

def get_sbert_features(texts):
    """Returns SBERT embeddings (Dense)"""
    model = SentenceTransformer('all-MiniLM-L6-v2')
    return model.encode(texts, show_progress_bar=False, batch_size=32).astype(np.float32)


def run_hybrid_experiment(filepath, dataset_name, embedding_type, get_embedding_func):
    print(f"\n{'='*60}")
    print(f" HYBRID EXPERIMENT: {embedding_type} + TF-IDF - {dataset_name.upper()}")
    print(f"{'='*60}")
    
    df = pd.read_csv(filepath)
    texts = df['TITLE'].values
    labels = df['PUBLISHER'].values
    
    print(f"    Extracting TF-IDF features...")
    X_tfidf = get_tfidf_features(texts)
    
    print(f"    Computing {embedding_type} features...")
    X_emb = get_embedding_func(texts)
    
   
    X_emb_sparse = sp.csr_matrix(X_emb)
    X_hybrid = sp.hstack([X_tfidf, X_emb_sparse])
    print(f"    Combined Shape: {X_hybrid.shape} (Sparse)")
    
    scaler = StandardScaler(with_mean=False) 
    X_scaled = scaler.fit_transform(X_hybrid)
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        'Linear SVM': LinearSVC(random_state=42, max_iter=2000, class_weight='balanced'),
        'SGDClassifier': SGDClassifier(loss='log_loss', random_state=42, max_iter=1000, class_weight='balanced')
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []
    
    print(f"\n    Training {embedding_type} Hybrid models...\n")
    for name, clf in models.items():
        try:
            cv_acc = cross_val_score(clf, X_scaled, labels, cv=cv, scoring='accuracy').mean()
            cv_f1 = cross_val_score(clf, X_scaled, labels, cv=cv, scoring='f1_macro').mean()
            results.append({'Model': name, 'CV_Accuracy': cv_acc, 'CV_F1_Macro': cv_f1})
            print(f"    {name:<25} | Acc: {cv_acc:.4f} | F1: {cv_f1:.4f}")
        except Exception as e:
            print(f"   ⚠️ {name} failed: {e}")
            
    res_df = pd.DataFrame(results)
    best_f1 = res_df['CV_F1_Macro'].max()
    
    print(f"\n   🏆 Best F1 for {embedding_type} Hybrid ({dataset_name}): {best_f1:.4f}")
    return res_df, best_f1


all_results = {}

experiments = {
    'Word2Vec': get_word2vec_features,
    'SBERT': get_sbert_features
}

for dataset in ['clean_duo_data.csv', 'clean_trio_data.csv']:
    d_name = dataset.replace('.csv', '').split('_')[1].upper()
    print(f"\n Starting experiments for: {d_name}")
    
    for emb_name, emb_func in experiments.items():
        try:
            df_res, best_f1 = run_hybrid_experiment(dataset, d_name, emb_name, emb_func)
            all_results[f"{d_name}_{emb_name}"] = best_f1
        except Exception as e:
            print(f" Failed {d_name} {emb_name}: {e}")


print("\n\n" + "="*60)
print(" FINAL HYBRID COMPARISON SUMMARY")
print("="*60)
summary = []
for key, f1 in all_results.items():
    dset, emb = key.split('_')
    summary.append({'Dataset': dset, 'Embedding': emb, 'Best_F1_Macro': f1})

final_df = pd.DataFrame(summary)
print(final_df.to_string(index=False))


 Starting experiments for: DUO

 HYBRID EXPERIMENT: Word2Vec + TF-IDF - DUO
    Extracting TF-IDF features...
    Computing Word2Vec features...
    Combined Shape: (3073, 2600) (Sparse)

    Training Word2Vec Hybrid models...

    Logistic Regression       | Acc: 0.7943 | F1: 0.7757
    Linear SVM                | Acc: 0.7784 | F1: 0.7588
    SGDClassifier             | Acc: 0.7836 | F1: 0.7643

   🏆 Best F1 for Word2Vec Hybrid (DUO): 0.7757

 HYBRID EXPERIMENT: SBERT + TF-IDF - DUO
    Extracting TF-IDF features...
    Computing SBERT features...
    Combined Shape: (3073, 2884) (Sparse)

    Training SBERT Hybrid models...

    Logistic Regression       | Acc: 0.8174 | F1: 0.8016
    Linear SVM                | Acc: 0.8002 | F1: 0.7819
    SGDClassifier             | Acc: 0.8064 | F1: 0.7906

   🏆 Best F1 for SBERT Hybrid (DUO): 0.8016

 Starting experiments for: TRIO

 HYBRID EXPERIMENT: Word2Vec + TF-IDF - TRIO
    Extracting TF-IDF features...
    Computing Word2Vec features...
